# Document loaders in langchain

LangChain community has integrations - https://docs.langchain.com/oss/javascript/integrations/providers/overview

This integration helps us to read any kind of data from pdf, csv, json, etc files

# Sample Demo of reading a text file using the langchain document reader

In [ ]:
# install langchain_community
!pip install langchain_community

In [ ]:
from langchain_community.document_loaders import  TextLoader 
from dotenv import load_dotenv
import os

load_dotenv()
KEY = os.getenv("OPENAI_API_KEY")

loader = TextLoader("data.txt")
documents = loader.load()

---
# Text Splitter and Chunking
1. We do split the long data into chuck (words with some length like 500 words as 1 chunk)
2. Each word is called as a token, so in one chunk you will have 500 token as per the above example
3. To safely store the data without messing with sequence, we tend to go with chunk overlap option. Typically we can chunk_overlap size = 50

Then the calculation will like this.
* total words of the document = 1950
* chunk size = 500
  * chunk-1 ==> 1 to 500
  * chunk-2 ==> 501 to 1000
  * chunk-2 ==> 1001 to 1500
  * chunk-1 ==> 1500 to 1950
* chunk overlap = 50
  * overlap from 500-25, 500+25 ==> 475-525
  * overlap from 1000-25, 1000+25 ==> 975-1025
  * overlap from 1500-25, 1500+25 ==> 1475-1525

  


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=750, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

In [ ]:
print(text_splitter)

---
# Embeddings

In [ ]:
pip install langchain_openai

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:


embeddings = OpenAIEmbeddings()

In [ ]:
print(embeddings)

---
# FAISS Vector Stores

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
# Vector store
vectorstore = FAISS.from_documents(docs, embeddings)

print(vectorstore)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
# llm 
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)


In [ ]:
# prompt
prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context:\n\n{context}\n\nQuestion: {question}"
)

In [ ]:
#  rag pipeline
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [43]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [44]:
# Query
query = "What is this document about?"
res = rag_chain.invoke(query)

print(res)

This document is about **_Alice’s Adventures in Wonderland_**, the 1865 children’s novel by **Lewis Carroll**.


---
# Without LCEL


In [ ]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_openai import OpenAI

prompt = PromptTemplate(
    input_variable=["topic"],
    template="Tell me a joke about {topic}"
)

llm = OpenAI()

chain = LLMChain(llm=llm, prompt=prompt)

response = chain.run("cats")

---
# With LCEL


In [ ]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_openai import OpenAI

prompt = PromptTemplate(
    input_variable=["topic"],
    template="Tell me a joke about {topic}"
)

llm = OpenAI()

chain = prompt | model

response = chain.run("cats")

---
# final program including langchain, vector db, and RAG system

In [45]:
# import statements
from langchain_community.document_loaders import  TextLoader 
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os

# Loading the API key
load_dotenv()
KEY = os.getenv("OPENAI_API_KEY")

# document file loading ==> text document
loader = TextLoader("data.txt")
documents = loader.load()


# text_splitter = CharacterTextSplitter(chunk_size=750, chunk_overlap=50)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Embeddings
embeddings = OpenAIEmbeddings()

# Vector store
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

# llm 
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)


# prompt
prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context:\n\n{context}\n\nQuestion: {question}"
)

#  rag pipeline
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Query
query = "What is this document about?"
res = rag_chain.invoke(query)

print(res)

This document is about **_Alice’s Adventures in Wonderland_**, the 1865 children’s novel by **Lewis Carroll**.


---
# Agentic AI + Agentic RAG

In [53]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langgraph.graph import StateGraph, END
from typing import TypedDict


from dotenv import load_dotenv
import os

# Loading the API key
load_dotenv()
KEY = os.getenv("OPENAI_API_KEY")

# document file loading ==> text document
loader = TextLoader("data.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Embeddings
embeddings = OpenAIEmbeddings()

# Vector store
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()


# llm 
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)


# prompt
prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context:\n\n{context}\n\nQuestion: {question}"
)

# Define graph
class GraphState(TypedDict):
    question: str
    context: str
    answer: str

# Nodes
def retrieve(state: GraphState):
    docs = retriever.invoke(state["question"])
    context = "\n\n".join(doc.page_content for doc in docs)
    return {"context": context}

def generate(state: GraphState):
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke(
        {
            "context": state["context"],
            "question": state["question"]
        }
    )
    return {"answer": answer}

# Build the graph
graph = StateGraph(GraphState)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

# compile the graph
app = graph.compile()

# run the query
query = "what is this document about?"

res = app.invoke({"question": query})

print(res["answer"])




This document is about **_Alice’s Adventures in Wonderland_**, the 1865 children’s novel by **Lewis Carroll**.
